In [ ]:
from alphagenome import colab_utils
from alphagenome.data import gene_annotation, transcript
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# Start = g.108,570,633
# End = g.108570,792
# TAAACTTGATGTCTAGGCCACTTCCTTTCTCTCGGGACCTACTTTTTCCATGTGTAACAAGGTGGAGAGAAGGGTATTGGACTCACAAAGACACACAACAGTAGTAATTTTATTCTTTCAAACCTTCTGATGAAGTTGTTTCTAGGATTACCGTGGCATA

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Iterable, List, Dict, Optional, Tuple
from alphagenome import colab_utils
from alphagenome.data import gene_annotation, transcript
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd

# Load the AlphaGenome API key from a file (alphagenome_api_key.txt)
from pathlib import Path

def _load_api_key(filename="alphagenome_api_key.txt"):
    env_key = os.environ.get("ALPHAGENOME_API_KEY")
    if env_key:
        return env_key.strip()
    for directory in [Path.cwd(), *Path.cwd().parents]:
        candidate = directory / filename
        if candidate.exists():
            return candidate.read_text().strip()
    raise FileNotFoundError(
        f"Could not find {filename}. Put your AlphaGenome API key on a single line "
        "in that file (it is gitignored), or set the ALPHAGENOME_API_KEY env var."
    )


API_KEY = _load_api_key()
dna_model = dna_client.create(API_KEY)

# Load metadata objects for human.
output_metadata = dna_model.output_metadata(
    organism=dna_client.Organism.HOMO_SAPIENS
)

# We first load up a GTF file containing gene and transcript locations as annotated by GENCODE (more information on GTF format here):

# The GTF file contains information on the location of all trancripts.
# Note that we use genome assembly hg38 for human.
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

# Set up transcript extractors using the information in the GTF file.
# Mane select transcripts consists of of one curated transcript per locus.
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_mane_select_transcript(gtf_transcripts)
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)


# Filter to protein-coding genes and highly supported transcripts.
gtf_transcript = gene_annotation.filter_transcript_support_level(
    gene_annotation.filter_protein_coding(gtf), ['1']
)

# Extractor for identifying transcripts in a region.
transcript_extractor = transcript.TranscriptExtractor(gtf_transcript)

# Also define an extractor that fetches only the longest transcript per gene.
gtf_longest_transcript = gene_annotation.filter_to_longest_transcript(
    gtf_transcript
)
longest_transcript_extractor = transcript.TranscriptExtractor(
    gtf_longest_transcript
)

# And then fetch the gene’s location as a genome.
# Interval object by passing either its gene_symbol (HGNC naming convention) or ENSEMBL gene_id
interval = gene_annotation.get_gene_interval(gtf, gene_symbol='COL4A5')

# Resize the interval to the acceptable range for the model
interval = interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

# Assumes these are already imported/initialized in your notebook:
# - genome (has genome.Variant)
# - dna_client (has OutputType and SEQUENCE_LENGTH_1MB)
# - dna_model (has predict_variant and score_variant)
# - transcript_extractor (has extract)
# - plot_components (has plot + track components)
# - variant_scorers (has RECOMMENDED_VARIANT_SCORERS + tidy_scores)


def _validate_dna(seq: str) -> str:
    seq = seq.strip().upper()
    bad = {b for b in seq if b not in {"A", "C", "G", "T"}}
    if bad:
        raise ValueError(f"DNA sequence contains invalid bases: {sorted(bad)}")
    return seq


def _all_substitutions(ref_base: str) -> List[str]:
    return [b for b in ("A", "C", "G", "T") if b != ref_base]


def _variant_tag(chrom: str, pos: int, ref: str, alt: str) -> str:
    # g.POSref>alt (with POS as integer, ref/alt as letters)
    return f"g.{pos}{ref}>{alt}"


def _safe_filter_ontology(df: pd.DataFrame, ontology_curie: Optional[str]) -> pd.DataFrame:
    if ontology_curie is None:
        return df
    if "ontology_curie" not in df.columns:
        return df.iloc[0:0]  # empty (ontology requested but not available)
    return df[df["ontology_curie"] == ontology_curie]


def _select_one_kidney_track(track_data_obj):
    """Keep a single positive-strand kidney track (fallback: first track)."""
    td = track_data_obj.filter_to_positive_strand()
    if td.num_tracks == 0:
        td = track_data_obj
    biosample = td.metadata.get("biosample_name", pd.Series([""] * td.num_tracks)).astype(str)
    kidney_mask = biosample.str.contains("kidney", case=False, na=False)
    if kidney_mask.any():
        idx = kidney_mask[kidney_mask].index[0]
        pos_idx = td.metadata.index.get_loc(idx)
        return td.select_tracks_by_index([pos_idx])
    return td.select_tracks_by_index([0])


def _select_one_kidney_junction(junction_data_obj, strand: str = "+"):
    """Keep a single kidney junction track (fallback: first track)."""
    jd = junction_data_obj.filter_to_strand(strand)
    if jd.num_tracks == 0:
        jd = junction_data_obj
    biosample = jd.metadata.get("biosample_name", pd.Series([""] * jd.num_tracks)).astype(str)
    kidney_mask = biosample.str.contains("kidney", case=False, na=False).values
    if kidney_mask.any():
        one_mask = np.zeros(jd.num_tracks, dtype=bool)
        one_mask[np.where(kidney_mask)[0][0]] = True
        return jd.filter_tracks(one_mask)
    one_mask = np.zeros(jd.num_tracks, dtype=bool)
    one_mask[0] = True
    return jd.filter_tracks(one_mask)


def run_alphagenome_saturation_mutagenesis(
    dna_seq: str,
    start_pos_1based: int,
    *,
    chrom: str = "chrX",
    gene_name: str = "COL4A5",
    gene_strand_match: bool = True,
    ontology_curie: str = "UBERON:0002113",
    output_dir: str | Path = "alphagenome_saturation",
    plot_title: str = "Predicted REF vs. ALT effects of variant in kidney tissue",
    plot_window_bp: int = 4500,
    plot_zoom_bp: int = 1000,
    transcript_strand: str = "+",
    save_png: bool = True,
    save_csv: bool = True,
    overwrite: bool = False,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    For each position in dna_seq, generate all 3 substitutions and:
      - predict_variant
      - save two plots per variant:
        - alphagenome_variant_<g.POSref>alt>_kidney.png (4500 bp window)
        - alphagenome_variant_<g.POSref>alt>_kidney_zoomed.png (1000 bp window)
      - score_variant -> tidy_scores -> filter to (gene_name, ontology) -> save CSV named: alphagenome_scores_<g.POSref>alt>.csv

    Returns a summary DataFrame with one row per variant and file paths/status.
    """
    dna_seq = _validate_dna(dna_seq)
    outdir = Path(output_dir)
    png_dir = outdir / "png"
    csv_dir = outdir / "csv"
    outdir.mkdir(parents=True, exist_ok=True)
    png_dir.mkdir(parents=True, exist_ok=True)
    csv_dir.mkdir(parents=True, exist_ok=True)

    # scorers (as in your snippet)
    scorers = [
        variant_scorers.RECOMMENDED_VARIANT_SCORERS["SPLICE_JUNCTIONS"],
        variant_scorers.RECOMMENDED_VARIANT_SCORERS["SPLICE_SITE_USAGE"],
        variant_scorers.RECOMMENDED_VARIANT_SCORERS["SPLICE_SITES"],
        # optional:
        # variant_scorers.RECOMMENDED_VARIANT_SCORERS["RNA_SEQ"],
    ]

    # plotting colors
    ref_alt_colors = {"REF": "dimgrey", "ALT": "red"}

    results: List[Dict[str, object]] = []

    for i, ref_base in enumerate(dna_seq):
        pos = start_pos_1based + i
        for alt_base in _all_substitutions(ref_base):
            tag = _variant_tag(chrom, pos, ref_base, alt_base)

            png_path = png_dir / f"alphagenome_variant_{tag}_kidney.png"
            png_zoomed_path = png_dir / f"alphagenome_variant_{tag}_kidney_zoomed.png"
            csv_path = csv_dir / f"alphagenome_scores_{tag}.csv"

            if not overwrite and ((save_png and png_path.exists() and png_zoomed_path.exists()) or (save_csv and csv_path.exists())):
                if verbose:
                    print(f"[SKIP] {tag} (exists)")
                results.append(
                    {
                        "variant": tag,
                        "chrom": chrom,
                        "pos": pos,
                        "ref": ref_base,
                        "alt": alt_base,
                        "png": str(png_path) if save_png else None,
                        "png_zoomed": str(png_zoomed_path) if save_png else None,
                        "csv": str(csv_path) if save_csv else None,
                        "status": "skipped_exists",
                        "error": None,
                    }
                )
                continue

            if verbose:
                print(f"[RUN]  {tag}")

            row: Dict[str, object] = {
                "variant": tag,
                "chrom": chrom,
                "pos": pos,
                "ref": ref_base,
                "alt": alt_base,
                "png": str(png_path) if save_png else None,
                "png_zoomed": str(png_zoomed_path) if save_png else None,
                "csv": str(csv_path) if save_csv else None,
                "status": "ok",
                "error": None,
            }

            try:
                # Build variant
                variant = genome.Variant(
                    chromosome=chrom,
                    position=pos,
                    reference_bases=ref_base,
                    alternate_bases=alt_base,
                )

                # Interval
                interval = variant.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)

                # Predict
                variant_output = dna_model.predict_variant(
                    interval=interval,
                    variant=variant,
                    requested_outputs=[
                        dna_client.OutputType.RNA_SEQ,
                        dna_client.OutputType.SPLICE_SITES,
                        dna_client.OutputType.SPLICE_SITE_USAGE,
                        dna_client.OutputType.SPLICE_JUNCTIONS,
                    ],
                    ontology_terms=[ontology_curie] if ontology_curie else None,
                )

                # Transcripts
                transcripts = transcript_extractor.extract(interval)

                ref_output = variant_output.reference
                alt_output = variant_output.alternate

                # Plot with same logic as cohort/genomAD notebook (single kidney track).
                if save_png:
                    ref_junction_one = _select_one_kidney_junction(ref_output.splice_junctions, strand=transcript_strand)
                    alt_junction_one = _select_one_kidney_junction(alt_output.splice_junctions, strand=transcript_strand)
                    ref_rna_one = _select_one_kidney_track(ref_output.rna_seq)
                    alt_rna_one = _select_one_kidney_track(alt_output.rna_seq)

                    plot = plot_components.plot(
                        [
                            plot_components.TranscriptAnnotation(transcripts),
                            plot_components.Sashimi(
                                ref_junction_one,
                                ylabel_template="Reference {biosample_name} ({strand})\n{name}",
                            ),
                            plot_components.Sashimi(
                                alt_junction_one,
                                ylabel_template="Alternate {biosample_name} ({strand})\n{name}",
                            ),
                            plot_components.OverlaidTracks(
                                tdata={
                                    "REF": ref_rna_one,
                                    "ALT": alt_rna_one,
                                },
                                colors=ref_alt_colors,
                                ylabel_template="RNA_SEQ: {biosample_name} ({strand})\n{name}",
                            ),
                        ],
                        interval=ref_output.rna_seq.interval.resize(plot_window_bp),
                        fig_width=8,
                        fig_height_scale=0.7,
                        hspace=0.20,
                        annotations=[plot_components.VariantAnnotation([variant])],
                        title=plot_title,
                        xlabel="ChrX genomic position (bp)",
                    )
                    plot.savefig(str(png_path), dpi=300, bbox_inches="tight")

                    plot_zoomed = plot_components.plot(
                        [
                            plot_components.TranscriptAnnotation(transcripts),
                            plot_components.Sashimi(
                                ref_junction_one,
                                ylabel_template="Reference {biosample_name} ({strand})\n{name}",
                            ),
                            plot_components.Sashimi(
                                alt_junction_one,
                                ylabel_template="Alternate {biosample_name} ({strand})\n{name}",
                            ),
                            plot_components.OverlaidTracks(
                                tdata={
                                    "REF": ref_rna_one,
                                    "ALT": alt_rna_one,
                                },
                                colors=ref_alt_colors,
                                ylabel_template="RNA_SEQ: {biosample_name} ({strand})\n{name}",
                            ),
                        ],
                        interval=ref_output.rna_seq.interval.resize(plot_zoom_bp),
                        fig_width=8,
                        fig_height_scale=0.7,
                        hspace=0.20,
                        annotations=[plot_components.VariantAnnotation([variant])],
                        title=plot_title,
                        xlabel="ChrX genomic position (bp)",
                    )
                    plot_zoomed.savefig(str(png_zoomed_path), dpi=300, bbox_inches="tight")

                # Score
                variant_scores = dna_model.score_variant(
                    interval=interval,
                    variant=variant,
                    variant_scorers=scorers,
                )

                df = variant_scorers.tidy_scores([variant_scores], match_gene_strand=gene_strand_match)

                # Filter to gene + ontology
                if "gene_name" in df.columns:
                    df = df[df["gene_name"] == gene_name]
                df = _safe_filter_ontology(df, ontology_curie)

                if save_csv:
                    df.to_csv(str(csv_path), index=False)

            except Exception as e:
                row["status"] = "error"
                row["error"] = repr(e)
                if verbose:
                    print(f"[ERROR] {tag}: {e}")

            results.append(row)

    return pd.DataFrame(results)


# ---- Example usage ----
# dna_seq_160 = "T" * 160  # replace with your real 160nt sequence
# summary = run_alphagenome_saturation_mutagenesis(
#     dna_seq_160,
#     start_pos_1based=108570633,
#     chrom="chrX",
#     gene_name="COL4A5",
#     ontology_curie="UBERON:0002113",
#     output_dir="alphagenome_saturation_COL4A5_intron6",
#     overwrite=False,
#     verbose=True,
# )
# summary.head()
# summary.to_csv("alphagenome_saturation_summary.csv", index=False)

In [ ]:
dna_seq_160 = "TAAACTTGATGTCTAGGCCACTTCCTTTCTCTCGGGACCTACTTTTTCCATGTGTAACAAGGTGGAGAGAAGGGTATTGGACTCACAAAGACACACAACAGTAGTAATTTTATTCTTTCAAACCTTCTGATGAAGTTGTTTCTAGGATTACCGTGGCATA" 

summary = run_alphagenome_saturation_mutagenesis(
    dna_seq_160,
    start_pos_1based=108570633,
    chrom="chrX",
    gene_name="COL4A5",
    ontology_curie="UBERON:0002113",
    output_dir="alphagenome_ISM_COL4A5_intron6",
    overwrite=False,
    verbose=True,
)

In [ ]:
## Test other sequence (COL4A5 intron 47, 123bp)
dna_seq_123 = "GCCAGAACTTCCAATACTATGTTGAATAGGAGTGAACTCAGGATTAAGAAACTCACTCAAAACCGCACAACTACATGGAAACTGTACCACATGCTCCTGAATGACTACTGGGTAAATAACGAA" 

summary = run_alphagenome_saturation_mutagenesis(
    dna_seq_123,
    start_pos_1based=108683908,
    chrom="chrX",
    gene_name="COL4A5",
    ontology_curie="UBERON:0002113",
    output_dir="alphagenome_ISM_COL4A5_intron47",
    overwrite=False,
    verbose=True,
)